# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Baselines

## load data

In [2]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/baselines'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'batch_size', 'patience', 'individual', 'train_epochs', 'lradj']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    model, individual = config['model'].values[0], config['individual'].values[0]
    if model == 'DLinear' and individual:
        result['model'].values[0] = 'DLinear_Ind'
    result.loc[:, metric_names] = metric[1], metric[0]
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df.to_csv(f"{save_root}/baseline_all_results.csv", index=False)

df.head(4)

,model,pred_len,data_id,learning_rate,batch_size,patience,individual,train_epochs,lradj,mse,mae,exp_dir
1039,Autoformer,96,ECL,0.0005,32,3,0,10,type1,0.188511,0.303782,/data/home/Licheng/workspace/TSF-PCA/results_P...
627,Autoformer,192,ECL,0.0005,32,3,0,10,type1,0.270894,0.370508,/data/home/Licheng/workspace/TSF-PCA/results_P...
1138,Autoformer,336,ECL,0.0005,32,3,0,10,type1,0.242762,0.352282,/data/home/Licheng/workspace/TSF-PCA/results_P...
89,Autoformer,720,ECL,0.0005,32,3,0,10,type1,0.294784,0.388354,/data/home/Licheng/workspace/TSF-PCA/results_P...


## analysis

In [3]:
min_mode = 'each'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'batch_size', 'patience', 'individual', 'train_epochs']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df2.to_csv(f'{save_root}/baselines_params.csv', index=False)

df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'
df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df2 = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2 = df2[columns]

df2.to_excel(f'{save_root}/baselines.xlsx')
df2.to_csv(f'{save_root}/baselines.csv')
df2

/tmp/ipykernel_3645157/2609304744.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model            Fredformer               FBM_L           iTransformer  \
                        mse       mae       mse       mae          mse   
data_id pred_len                                                         
ETTm1   96         0.326369  0.360869  0.349528  0.371773     0.337877   
        192        0.365194  0.382132  0.390812  0.392347     0.381666   
        336        0.395987  0.404369  0.422590  0.412849     0.426993   
        720        0.459217  0.444342  0.485965  0.448586     0.495750   
        Avg        0.386692  0.397928  0.412224  0.406389     0.410572   
ETTm2   96         0.177032  0.259926  0.181807  0.264386     0.181968   
        192        0.241639  0.299673  0.246169  0.304308     0.257024   
        336        0.301849  0.340199  0.306986  0.342475     0.319923   
        720        0.398819  0.396739  0.406963  0.397420     0.422518   
        Avg        0.279835  0.324134  0.285481  0.327147     0.295358   
ETTh1   96         0.377193  0.395888  0.383165  0.393112     0.385231   
        192        0.437019  0.425390  0.433448  0.422649     0.440491   
        336        0.485769  0.448580  0.474278  0.444622     0.479858   
        720        0.487772  0.467352  0.463483  0.461765     0.503538   
        Avg        0.446938  0.434302  0.438593  0.430537     0.452279   
ETTh2   96         0.293376  0.343882  0.288704  0.337401     0.301348   
        192        0.371951  0.391174  0.374276  0.389515     0.383043   
        336        0.420394  0.433487  0.413145  0.424124     0.424908   
        720        0.420675  0.439153  0.416531  0.436278     0.436269   
        Avg        0.376599  0.401924  0.373164  0.396829     0.386392   
ECL     96         0.161018  0.257591  0.197358  0.273776     0.150035   
        192        0.173585  0.269470  0.197128  0.276295     0.168114   
        336        0.194108  0.290397  0.211755  0.291585     0.182346   
        720        0.234672  0.319384  0.253368  0.324154     0.214451   
        Avg        0.190846  0.284211  0.214902  0.291452     0.178736   
Traffic 96         0.460576  0.327217  0.645200  0.382956     0.396554   
        192        0.469513  0.325797  0.597817  0.359186     0.415822   
        336        0.491848  0.338340  0.605160  0.361769     0.429438   
        720        0.521307  0.352766  0.642555  0.381331     0.462221   
        Avg        0.485811  0.336030  0.622683  0.371311     0.426009   
Weather 96         0.180192  0.219763  0.194073  0.233393     0.171432   
        192        0.222379  0.257628  0.240332  0.270332     0.246395   
        336        0.283492  0.300976  0.291561  0.306318     0.296230   
        720        0.357656  0.348500  0.363714  0.352717     0.362297   
        Avg        0.260930  0.281717  0.272420  0.290690     0.269089   
PEMS03  12         0.080544  0.191345  0.116822  0.225549     0.072155   
        24         0.120853  0.239561  0.233354  0.321429     0.104247   
        36         0.180330  0.292452  0.378731  0.419020     0.137279   
        48         0.201483  0.315661  0.534499  0.514812     0.173674   
        Avg        0.145803  0.259755  0.315851  0.370203     0.121839   
PEMS08  12         0.090728  0.198661  0.120719  0.231072     0.083674   
        24         0.138053  0.245493  0.232158  0.326810     0.123175   
        36         0.199214  0.303334  0.376393  0.425502     0.169713   
        48         0.254821  0.338272  0.543686  0.525740     0.218126   
        Avg        0.170704  0.271440  0.318239  0.377281     0.148672   

model                          FreTS            TimesNet                MICN  \
                       mae       mse       mae       mse       mae       mse   
data_id pred_len                                                               
ETTm1   96        0.372283  0.341839  0.374557  0.367581  0.394326  0.319464   
        192       0.396060  0.384536  0.400407  0.406042  0.409226  0.364399   
        336       0.423962  0.415986  0.420578 

# Finetune

## load data

In [77]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/finetune'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'auxi_type', 'auxi_mode', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df.to_csv(f"{save_root}/finetune_all_results.csv", index=False)

df.head(4)

KeyboardInterrupt: 

## preprocess

In [72]:
df3 = df.copy()
# df3 = df3[
#     ((df3.model == 'FBM_L') & (df3.data_id.isin(['ETTh1_PCA', 'ETTh2_PCA', 'ETTm2_PCA']))) | \
#     ((df3.model == 'MICN') & (df3.data_id.isin(['ETTm1_PCA', 'PEMS03_PCA']))) | \
#     ((df3.model == 'iTransformer') & (df3.data_id.isin(['ECL_PCA', 'Traffic_PCA', 'PEMS08_PCA']))) | \
#     ((df3.model == 'FreTS') & (df3.data_id.isin(['Weather_PCA'])))
# ]
df3 = df3[
    (df3.seq_len == 96) & \
    df3.data_id.isin(['ETTh1_PCA', 'ETTh2_PCA', 'ETTm1_PCA', 'ETTm2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']) & \
    (df3.auxi_loss == 'MAE') & \
    (df3.reinit == 1) & \
    (df3.use_weights == 0) & \
    (df3.pca_dim == 'T')
]

columns = ['model', 'pred_len', 'data_id', 'learning_rate', 'alpha', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'auxi_type', 'auxi_mode', 'lradj', 'patience', 'train_epochs', 'exp_dir']
df3.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
df3 = df3[columns + ['mse', 'mae']]

## summary

In [73]:
min_mode = 'each'

df2 = df3.copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df2 = df2.groupby(columns).filter(is_full_group)
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'auxi_type', 'auxi_mode', 'lradj', 'patience', 'train_epochs', 'exp_dir']
df2 = df2[columns]

dst_order = ['ETTm1_PCA', 'ETTm2_PCA', 'ETTh1_PCA', 'ETTh2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)
df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df2.dropna(inplace=True, thresh=5)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
os.makedirs(save_root, exist_ok=True)
df2.to_excel(f'{save_root}/best_finetune_full_{min_mode}.xlsx')
df2.to_csv(f'{save_root}/best_finetune_full_{min_mode}.csv', index=False)
df2

/tmp/ipykernel_3645157/3917441254.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,learning_rate,alpha,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,auxi_type,auxi_mode,lradj,patience,train_epochs,exp_dir
84,Fredformer,96,ETTm1_PCA,0.321138,0.357436,0.000500,0.400,T,1.0,0.0,1.000,MAE,128.0,pca,basis,TST,10.00,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
85,Fredformer,192,ETTm1_PCA,0.359581,0.378119,0.000500,0.800,T,1.0,0.0,1.000,MAE,128.0,pca,basis,TST,10.00,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
86,Fredformer,336,ETTm1_PCA,0.389238,0.399987,0.000500,0.900,T,1.0,0.0,0.700,MAE,128.0,pca,basis,TST,10.00,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
87,Fredformer,720,ETTm1_PCA,0.447020,0.434891,0.000500,1.000,T,1.0,0.0,0.800,MAE,128.0,pca,basis,TST,10.00,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
152,Fredformer,Avg,ETTm1_PCA,0.379244,0.392608,0.000500,0.775,NaN,1.0,0.0,0.875,NaN,128.0,NaN,NaN,NaN,10.00,100.0,NaN
36,FBM_L,96,ETTm1_PCA,0.342714,0.364978,0.000500,1.000,T,1.0,0.0,1.000,MAE,32.0,pca,basis,type1,3.00,10.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
37,FBM_L,192,ETTm1_PCA,0.387283,0.389597,0.000100,1.000,T,1.0,0.0,1.000,MAE,32.0,pca,basis,type1,3.00,10.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
38,FBM_L,336,ETTm1_PCA,0.419475,0.410351,0.001000,1.000,T,1.0,0.0,1.000,MAE,32.0,pca,basis,type1,3.00,10.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
39,FBM_L,720,ETTm1_PCA,0.481819,0.445778,0.000100,1.000,T,1.0,0.0,1.000,MAE,32.0,pca,basis,type1,3.00,10.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
161,FBM_L,Avg,ETTm1_PCA,0.407823,0.402676,0.000425,1.000,NaN,1.0,0.0,1.000,NaN,32.0,NaN,NaN,NaN,3.00,10.0,NaN


## long term forecasting final results

In [74]:
df4 = df2.copy()

df4 = df4[
    ((df4.model == 'Fredformer') & (df4.data_id == 'ETTm1_PCA')) | \
    ((df4.model == 'Fredformer') & (df4.data_id == 'ETTm2_PCA')) | \
    ((df4.model == 'Fredformer') & (df4.data_id == 'ETTh1_PCA')) | \
    ((df4.model == 'Fredformer') & (df4.data_id == 'ETTh2_PCA')) | \
    ((df4.model == 'iTransformer') & (df4.data_id == 'ECL_PCA')) | \
    ((df4.model == 'iTransformer') & (df4.data_id == 'Traffic_PCA')) | \
    ((df4.model == 'FreTS') & (df4.data_id == 'Weather_PCA')) | \
    ((df4.model == 'MICN') & (df4.data_id == 'PEMS03_PCA')) | \
    ((df4.model == 'iTransformer') & (df4.data_id == 'PEMS08_PCA'))
]

os.makedirs(save_root, exist_ok=True)
df4.to_excel(f'{save_root}/best_finetune_{min_mode}_params.xlsx')
df4.to_csv(f'{save_root}/best_finetune_{min_mode}_params.csv', index=False)
df4

,model,pred_len,data_id,mse,mae,learning_rate,alpha,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,auxi_type,auxi_mode,lradj,patience,train_epochs,exp_dir
84,Fredformer,96,ETTm1_PCA,0.321138,0.357436,0.000500,0.400,T,1.0,0.0,1.000,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
85,Fredformer,192,ETTm1_PCA,0.359581,0.378119,0.000500,0.800,T,1.0,0.0,1.000,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
86,Fredformer,336,ETTm1_PCA,0.389238,0.399987,0.000500,0.900,T,1.0,0.0,0.700,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
87,Fredformer,720,ETTm1_PCA,0.447020,0.434891,0.000500,1.000,T,1.0,0.0,0.800,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
152,Fredformer,Avg,ETTm1_PCA,0.379244,0.392608,0.000500,0.775,NaN,1.0,0.0,0.875,NaN,128.0,NaN,NaN,NaN,10.0,100.0,NaN
88,Fredformer,96,ETTm2_PCA,0.172235,0.251189,0.000050,1.000,T,1.0,0.0,0.800,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
89,Fredformer,192,ETTm2_PCA,0.235445,0.293936,0.000500,1.000,T,1.0,0.0,1.000,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
90,Fredformer,336,ETTm2_PCA,0.293289,0.332727,0.000050,1.000,T,1.0,0.0,1.000,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
91,Fredformer,720,ETTm2_PCA,0.387706,0.389056,0.000100,1.000,T,1.0,0.0,0.900,MAE,128.0,pca,basis,TST,10.0,100.0,/data/home/Licheng/workspace/TSF-PCA/results_P...
153,Fredformer,Avg,ETTm2_PCA,0.272169,0.316727,0.000175,1.000,NaN,1.0,0.0,0.925,NaN,128.0,NaN,NaN,NaN,10.0,100.0,NaN


# Merge

## load data

In [75]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes = pd.read_csv(f'{save_root}/best_finetune_each_params.csv')

model_list = ['Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'FEDformer', 'Autoformer', 'Transformer']

base = baselines.copy()
base = base[base.model.isin(model_list)]

best = finetunes.copy()
best['model'] = 'PDF'
best['data_id'] = best['data_id'].str.replace('_PCA', '')
best = best[best.pred_len != 'Avg']

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae']
df2 = pd.concat([base[columns], best[columns]], ignore_index=True)
df2['pred_len'] = df2['pred_len'].astype(int)

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'FEDformer', 'Autoformer', 'Transformer']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'
df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df2 = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2 = df2[columns]

df2.round(3).to_csv(f'{save_root}/long_term_results.csv', float_format='%.3f')
df2.round(4)

/tmp/ipykernel_3645157/2837928657.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model                PDF         Fredformer         iTransformer          \
                     mse     mae        mse     mae          mse     mae   
data_id pred_len                                                           
ETTm1   96        0.3211  0.3574     0.3264  0.3609       0.3379  0.3723   
        192       0.3596  0.3781     0.3652  0.3821       0.3817  0.3961   
        336       0.3892  0.4000     0.3960  0.4044       0.4270  0.4240   
        720       0.4470  0.4349     0.4592  0.4443       0.4958  0.4626   
        Avg       0.3792  0.3926     0.3867  0.3979       0.4106  0.4137   
ETTm2   96        0.1722  0.2512     0.1770  0.2599       0.1820  0.2647   
        192       0.2354  0.2939     0.2416  0.2997       0.2570  0.3148   
        336       0.2933  0.3327     0.3018  0.3402       0.3199  0.3540   
        720       0.3877  0.3891     0.3988  0.3967       0.4225  0.4114   
        Avg       0.2722  0.3167     0.2798  0.3241       0.2954  0.3362   
ETTh1   96        0.3680  0.3908     0.3772  0.3959       0.3852  0.4053   
        192       0.4241  0.4220     0.4370  0.4254       0.4405  0.4368   
        336       0.4670  0.4414     0.4858  0.4486       0.4799  0.4572   
        720       0.4649  0.4631     0.4878  0.4674       0.5035  0.4916   
        Avg       0.4310  0.4293     0.4469  0.4343       0.4523  0.4477   
ETTh2   96        0.2816  0.3302     0.2934  0.3439       0.3013  0.3492   
        192       0.3590  0.3805     0.3720  0.3912       0.3830  0.3972   
        336       0.3938  0.4138     0.4204  0.4335       0.4249  0.4323   
        720       0.4001  0.4270     0.4207  0.4392       0.4363  0.4485   
        Avg       0.3586  0.3879     0.3766  0.4019       0.3864  0.4068   
ECL     96        0.1449  0.2348     0.1610  0.2576       0.1500  0.2415   
        192       0.1589  0.2487     0.1736  0.2695       0.1681  0.2591   
        336       0.1731  0.2645     0.1941  0.2904       0.1823  0.2744   
        720       0.2033  0.2920     0.2347  0.3194       0.2145  0.3035   
        Avg       0.1701  0.2600     0.1908  0.2842       0.1787  0.2696   
Traffic 96        0.3926  0.2650     0.4606  0.3272       0.3966  0.2712   
        192       0.4098  0.2750     0.4695  0.3258       0.4158  0.2789   
        336       0.4209  0.2802     0.4918  0.3383       0.4294  0.2860   
        720       0.4507  0.2985     0.5213  0.3528       0.4622  0.3028   
        Avg       0.4185  0.2797     0.4858  0.3360       0.4260  0.2847   
Weather 96        0.1692  0.2185     0.1802  0.2198       0.1714  0.2105   
        192       0.2102  0.2575     0.2224  0.2576       0.2464  0.2783   
        336       0.2586  0.2971     0.2835  0.3010       0.2962  0.3130   
        720       0.3271  0.3487     0.3577  0.3485       0.3623  0.3528   
        Avg       0.2413  0.2805     0.2609  0.2817       0.2691  0.2886   
PEMS03  12        0.0699  0.1762     0.0805  0.1913       0.0722  0.1792   
        24        0.0871  0.1981     0.1209  0.2396       0.1042  0.2167   
        36        0.1055  0.2194     0.1803  0.2925       0.1373  0.2513   
        48        0.1237  0.2383     0.2015  0.3157       0.1737  0.2847   
        Avg       0.0965  0.2080     0.1458  0.2598       0.1218  0.2330   
PEMS08  12        0.0811  0.1831     0.0907  0.1987       0.0837  0.1870   
        24        0.1173  0.2178     0.1381  0.2455       0.1232  0.2269   
        36        0.1569  0.2528     0.1992  0.3033       0.1697  0.2681   
        48        0.2071  0.2941     0.2548  0.3383       0.2181  0.3061   
        Avg       0.1406  0.2370     0.1707  0.2714       0.1487  0.2470   

model              FreTS         TimesNet            MICN            TiDE  \
                     mse     mae      mse     mae     mse     mae     mse   
data_id pred_len                                                            
ETTm1   96        0.3418  0.3746   0.3676  0.3943  0.3195  0.3655  0.3529   
        192       0.3845  0.4004   0.4060  

In [76]:
df2a = df2_avg.copy()


df2a = df2a.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2a.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2a = df2a[columns]

df2a.round(3).to_csv(f'{save_root}/long_term_results_avg.csv', float_format='%.3f')
df2a

model                  PDF           Fredformer           iTransformer  \
                       mse       mae        mse       mae          mse   
data_id pred_len                                                         
ETTm1   Avg       0.379244  0.392608   0.386692  0.397928     0.410572   
ETTm2   Avg       0.272169  0.316727   0.279835  0.324134     0.295358   
ETTh1   Avg       0.430996  0.429305   0.446938  0.434302     0.452279   
ETTh2   Avg       0.358642  0.387885   0.376599  0.401924     0.386392   
ECL     Avg       0.170051  0.259995   0.190846  0.284211     0.178736   
Traffic Avg       0.418511  0.279659   0.485811  0.336030     0.426009   
Weather Avg       0.241251  0.280469   0.260930  0.281717     0.269089   
PEMS03  Avg       0.096519  0.208009   0.145803  0.259755     0.121839   
PEMS08  Avg       0.140582  0.236964   0.170704  0.271440     0.148672   

model                          FreTS            TimesNet                MICN  \
                       mae       mse       mae       mse       mae       mse   
data_id pred_len                                                               
ETTm1   Avg       0.413716  0.413916  0.421182  0.438483  0.430418  0.396142   
ETTm2   Avg       0.336203  0.315991  0.364634  0.301994  0.334121  0.308182   
ETTh1   Avg       0.447707  0.489336  0.473515  0.472139  0.463155  0.533093   
ETTh2   Avg       0.406793  0.523777  0.496169  0.409188  0.419780  0.619974   
ECL     Avg       0.269609  0.199420  0.287601  0.212249  0.305589  0.191564   
Traffic Avg       0.284707  0.538091  0.329537  0.631450  0.338492  0.528958   
Weather Avg       0.288630  0.248560  0.293011  0.271045  0.294599  0.263501   
PEMS03  Avg       0.232957  0.148539  0.260600  0.126363  0.230071  0.105705   
PEMS08  Avg       0.247042  0.174184  0.275201  0.151745  0.242988  0.152543   

model                           TiDE             DLinear           FEDformer  \
                       mae       mse       mae       mse       mae       mse   
data_id pred_len                                                               
ETTm1   Avg       0.421255  0.413294  0.407082  0.402787  0.406771  0.442489   
ETTm2   Avg       0.363925  0.285932  0.327573  0.342181  0.391771  0.307774   
ETTh1   Avg       0.519238  0.447741  0.434524  0.456173  0.453143  0.447485   
ETTh2   Avg       0.546394  0.378299  0.401092  0.528937  0.498962  0.451734   
ECL     Avg       0.301516  0.215127  0.291933  0.211973  0.300595  0.214211   
Traffic Avg       0.311916  0.623523  0.373185  0.624593  0.384049  0.639760   
Weather Avg       0.320877  0.272055  0.290650  0.265288  0.316614  0.326166   
PEMS03  Avg       0.222944  0.316045  0.370418  0.216076  0.322211  0.151928   
PEMS08  Avg       0.258241  0.318261  0.378022  0.248770  0.331627  0.225699   

model                      Autoformer           Transformer            
                       mae        mse       mae         mse       mae  
data_id pred_len                                                       
ETTm1   Avg       0.457431   0.525861  0.490807    0.798614  0.648424  
ETTm2   Avg       0.353655   0.314520  0.357685    1.662133  0.917055  
ETTh1   Avg       0.469822   0.476525  0.482849    0.983233  0.773957  
ETTh2   Avg       0.460887   0.447956  0.459721    2.688061  1.290601  
ECL     Avg       0.327979   0.249238  0.353731    0.265113  0.358376  
Traffic Avg       0.397865   0.661691  0.416094    0.691587  0.378910  
Weather Avg       0.372051   0.318987  0.364568    0.699104  0.600846  
PEMS03  Avg       0.274800   0.411156  0.474964    0.122424  0.226171  
PEMS08  Avg       0.311668   0.421538  0.455584    0.239983  0.260712